In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# !df -h /kaggle/working

In [ ]:
# !du -sh /kaggle/working/* | sort -rh

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Forces the script to only see one GPU

In [ ]:
# %%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
if major_version >= 8:
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
# Prevention for memory fragmentation



In [ ]:
max_seq_length = 2048 # Adjust based on legal document lengths
dtype = None # Auto-detection
load_in_4bit = True # Essential for T4 GPUs

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters for DAPT
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Higher rank is often better for DAPT to absorb new knowledge
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

# Load your uploaded Parquet file
# /kaggle/input/datasets/rehanfargose/sc-dataset-1990-2025
dataset = load_dataset("parquet", data_files={"train": "../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_1990-2025.parquet"}, split="train")

def formatting_prompts_func(examples):
    # 'full_text' is the column name in parquet
    return { "text" : [t + tokenizer.eos_token for t in examples["full_text"]] }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments



In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, 
    dataset_batch_size = 100,
    args = TrainingArguments(
        per_device_train_batch_size = 2, # Keep at 1
        gradient_accumulation_steps = 4, # Increase if you reduce batch size
        warmup_steps = 50,
        # Re-calculate steps based on new batching logic
        # max_steps = (len(dataset) // (1 * 8)), 
        max_steps = 100, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        gradient_checkpointing = True,
        weight_decay = 0.01,
        save_strategy = "no",
        report_to = "none",
    ),
)

In [ ]:
trainer.train()

In [ ]:
# Save the model after DAPT
# model.save_pretrained_merged("Deepseek_R1_8B_DAPT", tokenizer, save_method = "merged_16bit")

In [ ]:
# Merge to 16bit and save as GGUF for Ollama
model.save_pretrained_gguf("my_dapt_model", tokenizer, quantization_method = "q4_k_m")


In [ ]:
# # To save lora adapters
# model.save_pretrained("Deepseek_R1_8B_DAPT_LoRA")
# tokenizer.save_pretrained("Deepseek_R1_8B_DAPT_LoRA")

In [ ]:
# import shutil
# from IPython.display import FileLink

# # Zip small adapter folder
# shutil.make_archive("Deepseek_R1_8B_DAPT_LoRA", 'zip', "Deepseek_R1_8B_DAPT_LoRA")

# # Generate the download link
# FileLink(r'Deepseek_R1_8B_DAPT_LoRA.zip')